# Camada Gold

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalog = "rocket_lab"

silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"

gold_schema_name = "vcredit_gold"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

### 1) Chamados Custos

In [0]:
chamados = spark.table(f"{silver_schema}.chamados")
custos   = spark.table(f"{silver_schema}.custos")
canais   = spark.table(f"{silver_schema}.canais")
motivos  = spark.table(f"{silver_schema}.motivos")

dim_chamado = (
    chamados
    .withColumn("sk_chamado", F.sha2(F.col("id_chamado").cast("string"), 256))
    .select("sk_chamado", "id_chamado", "motivo", "canal", "resolvido")
)

dim_custos = (
    custos
    .select("id_chamado", "custo")
)

dim_canais = (
    canais
    .select("nome_canal", "tipo_canal", "status_canal")
)

dim_motivos = (
    motivos
    .select("nome_motivo", "categoria", "criticidade", "categoria_imputada")
)

chamados_custos = (
    dim_chamado
    .join(dim_custos, on="id_chamado", how="left")
    .join(dim_canais, dim_chamado.canal == dim_canais.nome_canal, how="left")
    .join(dim_motivos, dim_chamado.motivo == dim_motivos.nome_motivo, how="left")
    .select("sk_chamado", "motivo", "canal", "resolvido", "custo", "nome_canal", "tipo_canal", "status_canal", "nome_motivo", "categoria", "criticidade", "categoria_imputada")
)

 #idempotente
(
    chamados_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.chamados_custos")
)

display(chamados_custos.orderBy("sk_chamado"))

### 2) Perguntas do Case

In [0]:
spark.table(f"{gold_schema}.chamados_custos").createOrReplaceTempView("fato_custos")

### Quantidade de problemas simples resolvidos em canais complexos e suas implicações

* Total e custo médio de referência: calcula o total de chamados resolvidos de baixa criticidade e o custo médio dos chamados de baixa criticidade atendidos pelo autosserviço.

* Quantidade e participação percentual: contabiliza os chamados resolvidos por cada canal humano e calcula sua porcentagem em relação ao total de chamados resolvidos de baixa criticidade.

* Custo total e custo excedente: soma os custos dos atendimentos humanos e estima o custo excedente em comparação ao cenário em que os mesmos chamados fossem atendidos pelo autosserviço.

* Filtros e organização: considera apenas chamados resolvidos, de baixa criticidade e atendidos pelo canal humano. Agrupa os resultados por canal e os ordena pelo custo excedente, do maior para o menor.

In [0]:
display(
    spark.sql("""
        WITH total_chamados_simples AS (
            SELECT count(*) AS total
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND f.criticidade = 'baixa'
        ),
        custo_medio_canal_simples AS (
            SELECT round(sum(f.custo) / count(*), 2) AS custo_medio
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND f.tipo_canal = 'autosservico'
            AND f.criticidade = 'baixa'
        )
        SELECT f.nome_canal AS Nome_canal, count(*) AS Qtd_Chamadas,
        round(100.0*count(*) / NULLIF((SELECT total FROM total_chamados_simples), 0), 2) AS Porcentagem,
        sum(f.custo) AS Custo_Total, 
        (sum(f.custo) - round(count(*) * (SELECT custo_medio FROM custo_medio_canal_simples), 2)) AS Custo_Excedente
        FROM fato_custos f
        WHERE f.resolvido = 'sim' AND f.tipo_canal = 'humano'
        AND f.criticidade = 'baixa'
        GROUP BY f.nome_canal
        ORDER BY Custo_Excedente DESC
    """)
)

### Quantidade de problemas complexos resolvidos em canais simples e suas implicações

* Total e custo médio de referência: calcula o total de chamados resolvidos de média e alta criticidade e o custo médio desses chamados quando atendidos pelo canal humano.

* Quantidade e participação percentual: contabiliza os chamados complexos resolvidos por cada canal de autosserviço e calcula sua porcentagem em relação ao total de chamados resolvidos de média e alta criticidade.

* Custo total e custo salvo: soma os custos dos atendimentos por autosserviço e estima a economia em comparação ao custo que esses mesmos chamados teriam se fossem atendidos pelo canal humano, utilizando o custo médio humano como referência.

* Filtros e organização: considera apenas chamados resolvidos, de média ou alta criticidade e atendidos pelo autosserviço. Agrupa os resultados por canal e os ordena pelo custo salvo, do maior para o menor.

Objetivo geral: identificar os canais de autosserviço que resolvem chamados complexos e estimar a economia potencial em comparação ao atendimento humano.

In [0]:
display(
    spark.sql("""
        WITH total_chamados_complexas AS (
            SELECT count(*) AS total
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND (f.criticidade = 'alta' OR f.criticidade = 'media')
        ),
        custo_medio_canal_complexo AS (
            SELECT round(sum(f.custo) / count(*), 2) AS custo_medio
            FROM fato_custos f
            WHERE f.resolvido = 'sim' AND f.tipo_canal = 'humano'
            AND (f.criticidade = 'alta' OR f.criticidade = 'media')
        )
        SELECT f.nome_canal AS Nome_canal, count(*) AS Qtd_Chamadas,
        round(100.0*count(*) /  NULLIF((SELECT total FROM total_chamados_complexas), 0), 2) AS Porcentagem,
        sum(f.custo) AS Custo_Total, 
        (round(count(*) * (SELECT custo_medio FROM custo_medio_canal_complexo), 2) - sum(f.custo)) AS Custo_Salvo
        FROM fato_custos f
        WHERE f.resolvido = 'sim' AND f.tipo_canal = 'autosservico'
        AND (f.criticidade = 'alta' OR f.criticidade = 'media')
        GROUP BY f.nome_canal
        ORDER BY Custo_Salvo DESC
    """)
)